<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 4 - Fifty Shades of Bandit</h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                Notebook author: <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

# Objectives
The objective of this TD is to revisit the 2-armed Bandit task that we have seen in TD 1:
- This time, we will write a class in order to generate the bandit trajectories according to a set of initial parameters
- We will explore different of types of bandit tasks:
    - Stationary
    - Reversal
    - Restless
- We will implement a basic RL model that can play this task and understand how its parameters change the performance on the different types of bandit tasks.
- We will write an optimization routine that optimizes a model's parameters in order to get the best performance on a given task

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pybads import BADS
from typing import Tuple

from tests import test_rl_model_policy, test_bandit_task_generate_trajectories, \
    test_reversal_bandit_task_generate_trajectories, test_rl_model_update, test_rl_model_simulate

## Stationary 2-armed bandit task
The basic 2-armed bandit task is stationary, i.e. the mean reward of each arm remains constant.

The reward yielded by an arms at any given trial, however, has a bit of variability and is drawn from a normal distribution with a given standard deviation. For example, at trial $t$, the reward given by arms $0$ and $1$ are given by:
$$r_t^{(0)} \sim \mathcal{N}(\mu^{(0)}_t, \sigma^2) $$
$$r_t^{(1)} \sim \mathcal{N}(\mu^{(1)}_t, \sigma^2) $$

### The `BanditTask` class
We will write the basic bandit task in an object-oriented way. This will allow us to implement basic functionalities now and extend the class later without having to reinvent the wheel.

The class comes with a constructor, two methods that you need to fill, and a few more that are already implemented for you.

📝 Complete the `generate_trajectories()` method, which creates and returns two NumPy arrays with dimensions (n_trials, 2):
- `means`: the mean reward of each arm (if the mean is constant, we just repeat it `n_trials` times)
- `rewards`: the actual rewards given by each arm at every trial. Rewards are drawn from a normal distribution centered on the mean of the trial, with a standard deviation specified in the parametrization of the BanditTask. We then need to clip reward values to make sure they are within the $[0, 1]$ range.

In [ ]:
class BanditTask:
    def __init__(self, init_0: float, init_1: float, std: float, n_trials: int):
        self.mean_0 = init_0
        self.mean_1 = init_1
        self.std = std
        self.n_trials = n_trials
        self.means, self.rewards = self.generate_trajectories()

    def generate_trajectories(self) -> Tuple[np.ndarray, np.ndarray]:
        # Complete this method

        means = np.zeros((self.n_trials, 2))
        means[:, 0] = self.mean_0
        # To do: set arm 1's means

        rewards = ... # To do: sample rewards from their distribution, then ensure they stay in the [0,1] range

        return means, rewards

    def plot(self):
        # Your code here
        ...

    def reset(self):
        self.means, self.rewards = self.generate_trajectories()
        return self

    def __iter__(self):
        for rewards in self.rewards:
            yield rewards

    def __repr__(self):
        return f"{self.__class__.__name__}(std={self.std:.2f})"

🌟Today's TD comes with unit tests, thanks to our sponsor DecVPN... 🙃

⚙️ More seriously, you can simply run the cell below and it will tell you if your implementation of `generate_trajectories()` is correct, and if not, it will tell you what the problem is.

In [ ]:
test_bandit_task_generate_trajectories(BanditTask)

We will also write a method to generate a graphical visualization of the task. This method will plot the mean and the rewards of each arm over time. It will be reused in all variations of the BanditTask.

📝Fill the `plot()` method in the `BanditTask` class such that:
- The trials are on the x axis
- The rewards are on the y axis
- Use a line plot for the means of each arm
- Use a dotted line plot for the actual rewards
- Try to use a distinct color for each arm
- Add a legend to the plot

📝 Create an instance of `BanditTask` with the following parameters, and call its `plot()` method:
- `init_0` = 0.3
- `init_1` = 0.7
- `std`: 0.15
- `n_trials` = 40

In [ ]:
# Fill this in
bandit_task = ...
bandit_task.plot()

Check with your neighbor if your graphs are similar.

## Reversal 2-armed bandit task
The 2-armed bandit with reversal is similar to the stationary bandit, except that we swap the means halfway through the trials (we could add more reversals and vary their frequency, but we will keep it simple for today).

For the object-oriented implementation, we will use **class inheritance**: `ReversalBanditTask` will inherit all of `BanditTask`'s methods, and we will only need to re-write the `generate_trajectories()` to implement the reversal.

💡_Note_: This is a common pattern in object-oriented programming, called **method overriding**. It allows us reuse the code from the parent class and only change the parts that need to be different.

📝Write the `generate_trajectories()` method in the `ReversalBanditTask` class:
- Generate the sequence of means for the two arms (like you did in `BanditTask`)
- Add the reversal halfway through the trials
- **Only then**, generate the rewards using the means and the standard deviation

In [ ]:
class ReversalBanditTask(BanditTask):
    def generate_trajectories(self):
        # Your code here
        ...

        
        return ..., ...

⚙️ You can use the unit tests below to check if your implementation is correct.

In [ ]:
test_reversal_bandit_task_generate_trajectories(ReversalBanditTask)

📝Create an instance of `ReversaBanditTask` with parameters of your choice and check that the plot looks correct.

In [ ]:
# Your code here
reversal_bandit_task = ...
reversal_bandit_task.plot()

# Implementing a simple RL model
Now it's time to implement our first RL model to play this task! We will implement a simple model that uses a softmax policy to choose between the two arms. The model has two parameters, which we have already played with:
- `learning_rate`: the learning rate of the model ($\in [0, 1]$)
- `temperature`: the temperature of the softmax policy ($\in [0, 1]$)

This time, you will implement the two main components of the model:
- The **policy**, which determines the probability of choosing each action
- The **learning rule**, which updates the value of the chosen action

The basic reinforcement learning model assumes that the agent tracks the value of each action through time. These estimations are called **q-values**. Each bandit arm has its corresponding q-value, which corresponds to the reward that the agent expects to receive from it.

Below is the RLModel class. Have a look, but **go read the instruction below** to see what needs to be done in exercises **A**, **B** and **C**.

In [ ]:
class RLModel:
    def __init__(self, learning_rate: float, temperature: float):
        self.learning_rate = learning_rate
        self.temperature = temperature
        self.q_values = np.array([0.5, 0.5]) # At t=0, both arms are assumed to have a value of 0.5

    def policy(self):
        # Complete this method (exercise A) - READ THE INSTRUCTION BELOW
        np.seterr(over='ignore') # Silence the numerical overflow warnings, because NumPy already handles them
        ############################################################
        prob = ... # Use the softmax rule to compute the probability of choosing action 1

        ############################################################
        np.seterr(over='warn') # Restore warnings
        return prob

    def update(self, action: int, reward: float):
        # Write this method (exercise B) - READ THE INSTRUCTION BELOW
        ############################################################
        ... # Update the q_value of the model for the action that was chosen (TD rule)

        ############################################################
        # No need to return anything

    def simulate(self, bandit_task):
        # Complete this method (exercise C) - READ THE INSTRUCTION BELOW

        # We initialize actions, probs and rewards as lists (we will append values for each trial in the loop)
        actions = []
        probs = []
        rewards = []
        for trial_rewards in bandit_task:
            # Note: trial_rewards is an array of shape (2,) containing the rewards of the two arms on each trial
            ############################################################
            prob = ... # Compute the probability of the model choosing action 1 (what method does this again?)
            action = ... # Choose an action according to the probability (you can use a binomial distribution)
            reward = ... # What reward is gained if we use this action?

            ... # Update the model with the information gained in this trial

            # Update the history of actions, probabilities and rewards
            actions.append(action)
            ... # fill this
            ... # fill this
            ###########################################################

        # To return the history, we transform actions, probs and rewards into NumPy arrays
        return np.array(actions, dtype=int), np.array(probs, dtype=float), np.array(rewards, dtype=float)

    def reset(self):
        self.q_values = np.array([0.5, 0.5])

    def __repr__(self):
        return f'RLModel(lr={self.learning_rate: .3f}, t={self.temperature: .3f})'

### Exercise A: Policy

At each trial, the agent chooses the action that has the highest q-value, but with some randomness which depends on the degree of exploration that it is willing to do. This is mediated by the **temperature** parameter, where a model with a temperature of 0 will not explore at all and be 100% greedy. We will use the **softmax** policy, given by the following formula:
$$
p_t = \frac{1}{1 + \exp(\frac{-(Q_t^1 - Q_t^0)}{\tau})}
$$
where:
- $p_t$ is the probability of choosing action 1 at trial $t$
- $Q_t^0$ and $Q_t^1$ are the q-values of actions 0 and 1 at trial $t$
- $\tau$ is the temperature

📝Implement the `policy()` method in the `RLModel` class. Basically, all you need to do is re-write the softmax equation in NumPy.

__Optional note__: If you run the policy with a temperature that is close to 0, you will get a numerical overflow in the exponential. This overflow is correctly handled by np.exp(), but it still raises a warning. The two lines of code in the `policy()` method are there for this reason.

⚙️ When you are done, you can run the cell below to test your implementation.

In [ ]:
test_rl_model_policy(RLModel)

### Exercise B: Learning rule

When an action is chosen, the agent gets a reward, and this reward is compared with what was expected (i.e. its q-value). The model can **update** the corresponding q-value to match more or less the actual reward. The magnitude of this update is mediated by the **learning rate**:
- With a learning rate of 0, no update is done.
- With a learning rate of 1, the model completely overwrites the q-value with the received reward.
- With a learning rate of 0.5, the model updates the q-value halfway between its previous value and the received reward.

You will see that it's not always a good idea to value estimates too boldly.

More formally, if we suppose that on trial $t$, the agent selected action $a$, the update rule is given by:
$$
\begin{align}
\delta_t &= r_t - Q_t^{(a)} \\
Q_{t+1}^{(a)} &= Q^{(a)}_t + \alpha \times \delta_t
\end{align}
$$
Where:
- $\delta_t$ is the **temporal difference error** (a.k.a. TD error): the difference between the expected and received reward
- $Q_t^{(a)}$ is the q-value associated with action $a$ at time $t$
- $\alpha$ is the learning rate

📝Implement the `update()` method in the `RLModel` class.

⚙️ Then you can run the cell below to test your implementation.

In [ ]:
test_rl_model_update(RLModel)

### Exercise C: Simulating the model
📝Implement the `simulate` method in the `RLModel` class. This method should simulate the model on a given `BanditTask` instance. It should return three NumPy arrays:
- the actions chosen by the model. Dimensions: (n_trials,)
- the probabilities of choosing action 1. Dimensions: (n_trials,)
- the rewards obtained by the model. Dimensions: (n_trials,)

⚙️ Then you can run the cell below to test your implementation.

In [ ]:
test_rl_model_simulate(RLModel)

## Simulating the RL Model on the Basic and Reversal 2-armed bandit tasks
Remember that **model simulation** means making the model play the task and recording its behavior. Before we can experiment with different task configurations, we need to write a reusable function that efficiently runs the simulation and creates a visualization of the outcome.

### Generic helper function 🔧
📝 Write a function `simulate_and_plot()` that will:
- **Simulate** the **model** on the given **bandit task** (both are passed as arguments)
- Print the **parameters** of the model (thanks to the RLModel's `__repr__` method, all you need to do is call `print(model)`)
- Print the **total reward** obtained on all trials of the task
- **Plot the bandit task** using its `plot()` method
- On top, plot the model's **probability curve** (policy $p_t$), i.e. the probability of choosing arm 1 at any trial $t$
- On top, plot the model's **chosen actions** in a scatter plot (y=0 if $a_t=0$, y=1 if $a_t=1$)
- Update the legend

In [ ]:
def simulate_and_plot(model: RLModel, task: BanditTask):
    actions, probs, rewards = ..., ..., ... # Simulate the model given as an argument
    ... # Print model parameters
    ... # Print total reward
    ... # Plot the task
    ... # Plot the model's probabilities over all trials
    ... # Plot the model's actions over all trials
    ... # Extra plot formatting

📝 Execute your function here to see if it works correctly.

In [ ]:
test_task = ... # Instantiate a bandit task (anything you want)
test_model = ... # Instantiate the model (with parameters of your choice)

simulate_and_plot(test_model, test_task)

### Basic Bandit Task with well-separated reward trajectories (Very easy task)
We will start with a version of the bandit task where it is very easy to learn which arm gives out the best reward. We will create an instance of the BanditTask and keep it for all exercises of this section **without resetting it** (so we can better compare the different models and the reward they get).


📝 Initialize a stationary bandit task (`BanditTask`) with the following parameters:
- `init_0` = 0.2
- `init_1` = 0.8
- `std`: sufficiently low so that the two reward trajectories don't overlap;
- `n_trials` = 40

In [ ]:
basic_task = ...

📝 Simulate and plot an RLModel with a typical temperature setting (0.1), and a very high learning rate.

__Note__: use the `simulate_and_plot` function that you just created

In [ ]:
# Your code here
...

📝 Now try the same, but with a very small learning rate.

In [ ]:
# Your code here
...

📝 Now try increasing the temperature

In [ ]:
# Your code here
...

💭Based on the experiments you just did, answer these two questions:
- Is it better to have a high or a low learning rate in this task? Does this parameter matter much here?
- What is the role of the temperature in this task? How useful is exploration in this task?

_Your answer here_

### Reversal Bandit task with well-separated reward trajectories
Now you will perform the the same three simulations you just did, but on the reversal bandit task.

📝 Initialize a reversal bandit task (`ReversalBanditTask`) with the same parameters:
- `init_0` = 0.2
- `init_1` = 0.8
- `std`: same as you used for the basic task;
- `n_trials` = 40

In [ ]:
reversal_bandit_task = ...

📝 Simulate and plot an RLModel with a typical temperature setting (0.1), and a very high learning rate on the reversal bandit task.

In [ ]:
# Your code here
...

📝 With a small learning rate

In [ ]:
# Your code here
...

📝 With a high temperature

In [ ]:
# Your code here
...

💭Answer these two questions:
- Is the role of the learning rate more important in the reversal task than in the basic task? Why?
- What about temperature?

_Your answer here_

### Reversal Bandit task with overlapping reward trajectories (harder)
📝 Now create a reversal bandit task with the following parameters:
- `init_0` = 0.4
- `init_1` = 0.6
- `std`: 0.15
- `n_trials` = 40

In [ ]:
# Your code here
reversal_overlap_task = ...

By trial and error, try to figure out what parameter setting would be best to get good performance on this task.
- Start with a typical temperature setting of 0.1
- Find the best learning rate to make the model learn the reversal correctly
- Then change the temperature to see if it helps

In [ ]:
# Your code here
...

## 💪 Extra step: Optimizing the parameters of the RL model
Now that we have seen how the parameters of the model can impact its performance, we will write an optimization routine that will find the best parameters for a given task. This means finding the RL parameters that allow the model to get the highest reward on the task.

### Evaluating model performance
We will start by writing a function that computes the average performance of the model on a number of blocks.

📝 Complete the `model_performance()` function which:
- takes a **task**, **model** and number of **blocks** as inputs
- for each block:
  - Reset the task to get new bandit reward trajectories (see the `reset` method)
  - Simulate the model on the new task block
  - record the total reward received in the block
- return the total reward averaged across blocks

In [ ]:
def model_performance(task, model: RLModel, n_blocks: int):
    np.random.seed(42) # We seed the random number generator to get reproducible results
    total_rewards = []
    for i in range(n_blocks):
        task.reset() # We reset the task at each block to get new trajectories
        model.reset() # We reset the model to return to q-values of 0.5
        ... # TO DO: Simulate the model on the task
        ... # TO DO: Record the total reward gained by the model on the block

    return ... # TO DO: Return the average total reward over all blocks

Test your function manually. It should return a single float number, reresenting the average total reward that a model could get on a task of 40 trials.

In [ ]:
# Your code here
...

### Optimization routine
Now let's write the code that will find the set of parameters that maximize the rewards a model can get. For this, we can use a standard optimizer that will do the job for us. We will use BADS (Bayes Adaptive Direct Search). All we need to do is provide it with details about the range of values that we want to explore with each parameter, and a cost function that it needs to minimize. It's standard for optimizers to **minimize**, so since we have just defined a `model_performance` that we wish to **optimize**, we will have to add a minus sign to turn it into a minimization problem.

📝 Complete the `optimize_rl_model` function:
1. Write the `cost_function`, that gives the "cost" associated with an RLModel with the given parameters:
    -  Instantiate an RLModel with the parameters given as arguments
    -  Compute its average total reward on **20 blocks**
    -  Return the cost as a value that we wish to **minimize**


2. Complete the settings of the BADS optimizer, which lists that provide a value for each parameter in the following order: \[lr, temp]. I let you think about what settings could be appropriate and we'll discuss it together. As a reference:
    - `x0` represent the typical value that you would expect for each parameter. It's the starting point of the optimization.
    - `lower_bounds` and `upper_bounds` specify the range of values that will be explored. In our case we have bounds of $[0, 1]$ for both parameters
    - `plausible_lower_bound` and `plausible_upper_bound` are more restrictive than the actual bounds and specify the range where we expect to find most solutions.

In [ ]:

def optimize_rl_model(task, n_blocks: int,
                     verbose: bool = True):
    
    ##########################################################################
    # In python we can define functions inside functions.
    # Here we define the cost function that the BADS optimizer will try to minimize
    # It takes as input the parameters of the model and returns the average total reward
    def cost_function(params):
        learning_rate, temperature = params
        # Your code here
        ...
        ...
        return ... 
    ############################################################################

    optimizer = BADS(
        fun=cost_function,
        x0=[..., ...], # Fill this [lr, temp],
        lower_bounds=[..., ...], # Fill this
        upper_bounds=[..., ...], # Fill this
        plausible_lower_bounds=[..., ...], # Fill this
        plausible_upper_bounds=[..., ...], # Fill this
        options={
            'display': 'iter' if verbose else 'off'
        }
    )
    result = optimizer.optimize()
    optimal_params = {
        'learning_rate': result.x[0],
        'temperature': result.x[1]
    }
    return RLModel(**optimal_params)

📝Execute your optimization function on 20 blocks of the `basic_task`

In [ ]:
test_task = BanditTask(0.2, 0.4, 0.15, 40) # Use this test_task to test your function

... # Your code here


### Visualization helper function
Before we start having fun optimizing the RLModel on a variety of tasks, let's write one last helper function that does the optimization for us and plots the behavior of the optimal model on a new block of the task so that we can visualize it.

📝 Write the function `optimize_simulate_and_plot` that will:
- Optimize the parameters of RLModel on the given bandit task on 20 blocks (you can also set `verbose`=False to hide the details of the optimization)
- Reset the task
- Simulate the optimal model on the task (just one new block)
- Plot the simulated trajectory (using the `simulate_and_plot` function you already wrote
- Print the optimal model parameters and the total reward if your `simulate_and_plot` function doesn't do it already

__Note__: Have you noticed that here the `optimize_and_plot` function does not take an instance of RLModel as an argument? Can you explain why?

In [ ]:
def optimize_simulate_and_plot(task):
    # Your code here
    ...

📝Try your function using the `test_task` instance

### Comparison of optimal parameters on various task types
Now that we have all the tools we need, let's find the optimal RLModel parameters automatically for a variety of bandit tasks.

#### Basic Bandit Task
Let's see what happens Find the optimal parameters for the basic 2-arm bandit task, with a low standard deviation and distant means (you can use the `basic_task` you instantiated before)

In [ ]:
# Your code here
...

### Reversal Bandit Task (easy)
Find the optimal parameters for the reversal 2-armed bandit task where the trajectories didn't overlap (`reversal_bandit_task`)

In [ ]:
# Your code here
...

### Reversal Bandit Task (hard)
Do the same with the one with overlapping trajectories (`reversal_overlap_task`)

In [ ]:
# Your code here
...

### More simulations
📝Now you can explore more and create different instances of the bandit task and pass them to the `optimize_and_plot` function:
- You can decide the values of `init_0`, `init_1` and `std`.
- You can keep 40 trials or change it
- Bear in mind that the more the trajectories of the two bandit arms are intertwined, the less trivial is the task, and hence the more interesting it gets.

Make experiments that will allow you to answer the questions given below.

In [ ]:
# Your code here
...

💭Based on your experiments, answer the following questions:
- What is the impact of the standard deviation on the optimal parameters?
- What happens if the trajectories of the arms are completely separated?

_Your answers here_

## 💪💪 Optional: Restless 2-armed bandit task
Now what happens if the means of the arms are not kept constant, and add a little bit of volatility?

The restless bandit task is a 2-armed bandit task where the means of the arms are drifting over time.

### RestlessBanditTask class
We will implement the `RestlessBanditTask` class, which will also inherit from `BanditTask`. We need to rewrite the `generate_trajectories` method. More specifically, the means have to be generated trial by trial, such that each value must depend on the previous one, with a drift determined by a normal distribution:
$$
\mu_t \sim \mathcal{N}(\mu_{t-1}, \text{drift}^2)
$$
Don't forget to also clip the means between 0 and 1

Generate the rewards using the std the same way as it was done before.

In [ ]:
class RestlessBanditTask(BanditTask):
    def __init__(self, init_0: float, init_1: float, std: float, drift: float, n_trials: int):
        self.init_0 = init_0
        self.init_1 = init_1
        self.std = std
        self.drift = drift
        self.n_trials = n_trials
        self.means, self.rewards = self.generate_trajectories()

    def generate_trajectories(self):
        ## Your code here
        # Generate the means
        ...


        # Generate the rewards
        ...
        
        return ...        

    def __repr__(self):
        return f"{self.__class__.__name__}(std={self.std:.2f}, drift={self.drift:.2f})"

### Explore different degrees of task volatility and stochasticity
The drift parameter determines the **volatility** of the task and the std parameter determines the **stochasticity**.

📝 Find what RL parameters are optimal on a high-volatility and low-stochasticity task

In [ ]:
# Your code here
...

📝 Do the same with a high-stochasticity and low-volatility task

In [ ]:
# Your code here
...

✍️ What do you conclude?

Your answer here.